In [1]:
import warnings
warnings.simplefilter(action='ignore')
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

In [3]:
def plot_distribution(df, attribute, coloring, run, relation, combination, acc):
    palette ={"nlr": "C0", "busco": "C1", "housekeeping": "C2", "all": "C3", "rest":"C3"}
    order=["nlr","busco","housekeeping"] if combination == "reference" else ["nlr","busco","housekeeping", "all"] if combination == "background" else ["nlr","busco","housekeeping", "rest"]
    sns.displot(df, x=attribute, hue = coloring, kind="kde",common_norm=False, palette=palette, hue_order=order)
    plt.savefig("{}/{}/{}/{}_pasa_augustus_{}_{}.png".format(run,acc,relation,acc,combination,attribute), bbox_inches='tight')
    plt.close()

In [4]:
def plot_categories(df, attribute, coloring, run, relation, combination, acc):
    palette ={"nlr": "C0", "busco": "C1", "housekeeping": "C2", "all": "C3", "rest":"C3"}
    order=["nlr","busco","housekeeping"] if combination == "reference" else ["nlr","busco","housekeeping", "all"] if combination == "background" else ["nlr","busco","housekeeping", "rest"]
    sns.histplot(df, x=attribute, hue =coloring, multiple= "dodge", stat="percent", common_norm=False, shrink = 0.8,  palette=palette, hue_order=order)
    plt.xticks(rotation=45, horizontalalignment='right')
    plt.savefig("{}/{}/{}/{}_pasa_augustus_{}_{}.png".format(run,acc,relation,acc,combination,attribute), bbox_inches='tight')
    plt.close()

# FULL RUN

In [10]:
accessions= ['at6923','at6929','at7143','at8285','at9104','at9336','at9503','at9578','at9744','at9762','at9806','at9830','at9847','at9852','at9879','at9883','at9900','col0']
#accessions= ['at6923']

runs=["full"]


for run in tqdm(runs):
    pasa_augustus_df = pd.DataFrame()
    pasa_subcat = pd.DataFrame()
    for acc in tqdm(accessions):
        # -------------0.5-----------
        # prepare ref gene sets:
        busco = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/augustus_mappings/busco_{acc}.txt", header=None)
        house = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/augustus_mappings/housekeeping_{acc}.txt", header=None)
        nlr = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/augustus_mappings/nlrs_{acc}.txt", header=None)
        
        busco["type"] = "busco"
        house["type"] = "housekeeping"
        nlr["type"] = "nlr"
        gene_type = pd.concat([busco,house,nlr])
        gene_type.columns = ["ref_isoform","type"]
        gene_type = gene_type.groupby("ref_isoform")["type"].apply(', '.join).reset_index()

        gene_type["ref_gene"] = gene_type["ref_isoform"].str.split(".").str[0]
        gene_type.loc[gene_type["type"] == "busco, nlr"] = "nlr"
        gene_type.loc[gene_type["type"] == "housekeeping, nlr"] = "nlr"
        gene_type.loc[gene_type["type"] == "busco, housekeeping"] = "housekeeping"
        
        # -------------0-------------
        # make directory
        os.makedirs("{}/{}/gene".format(run,acc), exist_ok=True)
        # -------------1-------------
        # prepare sqanti output
        sqanti = pd.read_csv("/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/pasa/output/augustus/{}/{}-augustus_updated_tama-vs-augustus_classification.txt".format(acc,acc), sep="\t")
        sub = sqanti[["isoform","exons","length","structural_category","associated_gene","associated_transcript","subcategory","ORF_seq"]]
        sub["gene"] = sub["isoform"].str.split(".").str[0]
        sub['accession'] = acc

        pasa_augustus_df = pd.concat([pasa_augustus_df, sub])

        # -------------6-------------
        # relate the gene type to their gene in sqanti
        gene_ref = pd.merge(left=sub, right=gene_type, how='left', left_on='associated_gene', right_on='ref_gene')
        # ------------6.1------------
        # basic analysis
        gene_nlr = gene_ref.loc[gene_ref["type"] == "nlr"]
        gene_busco = gene_ref.loc[gene_ref["type"] == "busco"]
        gene_house = gene_ref.loc[gene_ref["type"] == "housekeeping"]
        gene_all = pd.concat([gene_nlr,gene_busco,gene_house])
        # plot exon + length distribution
        plot_distribution(gene_all, "exons", "type", run, "gene", "reference", acc)
        plot_distribution(gene_all, "length", "type", run,  "gene", "reference", acc)
        # plot category and subcategory
        plot_categories(gene_all, "structural_category", "type", run, "gene", "reference", acc)
        plot_categories(gene_all, "subcategory", "type", run,  "gene", "reference", acc)
        # ------------6.2------------
        # all genes as background, reference genes on top
        gene_background = sub.copy()
        gene_background["ref_isoform"] = "None"
        gene_background["type"] = "all"
        gene_background["ref_gene"] = "None"
        gene_background_all = pd.concat([gene_all,gene_background], ignore_index=True)
        # plot exon + length distribution
        plot_distribution(gene_background_all, "exons", "type", run, "gene", "background", acc)
        plot_distribution(gene_background_all, "length", "type", run,  "gene", "background", acc)
        # plot category and subcategory
        plot_categories(gene_background_all, "structural_category", "type", run, "gene", "background", acc)
        plot_categories(gene_background_all, "subcategory", "type", run,  "gene", "background", acc)
        # ------------6.3------------
        # rest vs. busco vs. house vs nlr.
        gene_ava = gene_ref.fillna("rest")
        # plot exon + length distribution
        plot_distribution(gene_ava, "exons", "type", run, "gene", "ava", acc)
        plot_distribution(gene_ava, "length", "type", run,  "gene", "ava", acc)
        # plot category and subcategory
        plot_categories(gene_ava, "structural_category", "type", run, "gene", "ava", acc)
        plot_categories(gene_ava, "subcategory", "type", run,  "gene", "ava", acc)

        # -------------7-------------
        # prepare the subcategory dataframe:
        idx=sub["structural_category"].value_counts().index
        cat_df = pd.DataFrame({"accession":[acc]})
        for cat in idx:
            cat_df[cat]=sub["structural_category"].value_counts()[cat]
        test=cat_df.set_index("accession").T[acc].sum()
        cat_df["sum"] = cat_df.set_index("accession").T[acc].sum()
        pasa_subcat=pd.concat([pasa_subcat, cat_df], ignore_index=True).fillna(0)
        
    pasa_augustus_df = pasa_augustus_df[["gene","isoform","accession","associated_gene","associated_transcript","exons","length","structural_category","subcategory","ORF_seq"]]
    pasa_augustus_df.columns = ["gene","isoform","accession","associated_gene","associated_transcript","exons","length","structural_category","subcategory","Sqanti_ORF_seq"]
    if run == "full":
        pasa_augustus_df.to_pickle("./pasa_augustus_sqanti_{}.pkl".format(run))
        pasa_augustus_df.to_csv("./pasa_augustus_sqanti_{}.tsv".format(run), sep="\t")

    # run analysis over the full dataframes
    os.makedirs("{}/{}/gene".format(run,"all"), exist_ok=True)
    
    # basic
    all_gene_ref = pd.merge(left=pasa_augustus_df, right=gene_type, how='left', left_on='associated_gene', right_on='ref_gene')
    all_gene_nlr = all_gene_ref.loc[all_gene_ref["type"] == "nlr"]
    all_gene_busco = all_gene_ref.loc[all_gene_ref["type"] == "busco"]
    all_gene_house = all_gene_ref.loc[all_gene_ref["type"] == "housekeeping"]
    all_gene_all = pd.concat([all_gene_nlr,all_gene_busco,all_gene_house])
    # plot exon + length distribution
    plot_distribution(all_gene_all, "exons", "type", run, "gene", "reference", "all")
    plot_distribution(all_gene_all, "length", "type", run, "gene", "reference", "all")
    # plot category and subcategory
    plot_categories(all_gene_all, "structural_category", "type", run, "gene", "reference", "all")
    plot_categories(all_gene_all, "subcategory", "type", run,  "gene", "reference", "all")

    # background
    all_background = pasa_augustus_df.copy()
    all_background["ref_isoform"] = "None"
    all_background["type"] = "all"
    all_background["ref_gene"] = "None"
    all_background_all = pd.concat([all_gene_all,all_background], ignore_index=True)
    # plot exon + length distribution
    plot_distribution(all_background_all, "exons", "type", run, "gene", "background", "all")
    plot_distribution(all_background_all, "length", "type", run, "gene", "background", "all")
    # plot category and subcategory
    plot_categories(all_background_all, "structural_category", "type", run, "gene", "background", "all")
    plot_categories(all_background_all, "subcategory", "type", run,  "gene", "background", "all")
    
    # all vs all
    all_gene_ava = all_gene_ref.fillna("rest")
    # plot exon + length distribution
    plot_distribution(all_gene_ava, "exons", "type", run, "gene", "ava", "all")
    plot_distribution(all_gene_ava, "length", "type", run, "gene", "ava", "all")
    # plot category and subcategory
    plot_categories(all_gene_ava, "structural_category", "type", run, "gene", "ava", "all")
    plot_categories(all_gene_ava, "subcategory", "type", run,  "gene", "ava", "all")
    
    for cat in pasa_subcat.set_index("accession").columns:
        pasa_subcat["proportion"]=pasa_subcat[cat]/pasa_subcat["sum"]*100
        sns.barplot(data=pasa_subcat, x="accession", y="proportion")
        plt.xticks(rotation=45, horizontalalignment='right')
        plt.ylabel("{} [%]".format(cat))
        plt.savefig("{}/all/all_pasa_augustus_{}.png".format(run,cat), bbox_inches='tight')
        plt.close()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [02:09<00:00, 129.76s/it]
